In [66]:
# =========================
# 0) Auto-locate processed/ and create dataset folders
# =========================
from pathlib import Path
import os

def _find_processed_dir(start: Path, max_depth: int = 5) -> Path:
    # Common locations first
    common = [
        start / "processed",
        start / "data" / "processed",
    ]
    for p in common:
        if p.exists() and p.is_dir():
            return p.resolve()

    # Bounded BFS search (keeps it reasonably fast)
    start = start.resolve()
    q = [(start, 0)]
    seen = set()

    while q:
        cur, depth = q.pop(0)
        if cur in seen:
            continue
        seen.add(cur)

        cand = cur / "processed"
        if cand.exists() and cand.is_dir():
            return cand.resolve()

        if depth >= max_depth:
            continue

        try:
            for child in cur.iterdir():
                if child.is_dir() and not child.name.startswith("."):
                    # skip huge/common noise folders
                    if child.name.lower() in {"node_modules", ".git", "__pycache__", ".venv", "venv"}:
                        continue
                    q.append((child, depth + 1))
        except Exception:
            continue

    raise FileNotFoundError(
        f"Could not find a 'processed' folder starting from: {start} (max_depth={max_depth})."
    )

# Move 2 steps back from current working directory
cwd = Path.cwd().resolve()
repo_root_guess = cwd.parent.parent

PROCESSED_ROOT = _find_processed_dir(repo_root_guess)

# Create dataset folders if missing
DATASET_NAMES = ["coaid", "fakehealth", "fakenewsnet", "hover", "liar"]
for name in DATASET_NAMES:
    (PROCESSED_ROOT / name).mkdir(parents=True, exist_ok=True)

print("cwd:", cwd)
print("repo_root_guess (2 steps back):", repo_root_guess)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("Created/verified dataset folders:", ", ".join(DATASET_NAMES))


cwd: C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\unified_schema\After_datasets_ready
repo_root_guess (2 steps back): C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data
PROCESSED_ROOT: C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed
Created/verified dataset folders: coaid, fakehealth, fakenewsnet, hover, liar


In [67]:
# =========================
# 1) Config (Cell 2)
# =========================
from pathlib import Path

DATASET = "hover"              # coaid / fakehealth / ...
SPLIT = "true"                # true / false / mixed (whatever your filename uses)
THRESH_PARTIAL = 0.4
THRESH_FULL = 0.05

# Full input file (adjust name if needed)
FULL_PATH = PROCESSED_ROOT / f"_{DATASET}_split_tmp" / f"{DATASET}_{SPLIT}_full.parquet"

# Outputs
KEPT_PATH = PROCESSED_ROOT / DATASET / f"{DATASET}_{SPLIT}.parquet"             # main kept file
FILTERED_PATH = PROCESSED_ROOT / DATASET / f"{DATASET}_{SPLIT}_filtered.parquet" # audit file

print("FULL_PATH    =", FULL_PATH)
print("KEPT_PATH    =", KEPT_PATH)
print("FILTERED_PATH=", FILTERED_PATH)


FULL_PATH    = C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\_hover_split_tmp\hover_true_full.parquet
KEPT_PATH    = C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\hover\hover_true.parquet
FILTERED_PATH= C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\hover\hover_true_filtered.parquet


In [68]:
# =========================
# 2) Load full.parquet (Cell 3)
# =========================
import pyarrow.parquet as pq
import pandas as pd

if not FULL_PATH.exists():
    raise FileNotFoundError(f"Missing: {FULL_PATH}")

full = pq.read_table(str(FULL_PATH)).to_pandas()

required_cols = ["id", "claim_text", "article_text"]
missing = [c for c in required_cols if c not in full.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}. Found: {list(full.columns)}")

print("rows:", len(full))
print("cols:", len(full.columns))
full.head(3)

rows: 0
cols: 15


,dataset,id,claim_text,article_text,content_status,label_raw,label,label_confidence,label_mode,label_3way,label_bin,source_id,claim_norm_hash,lang,content_char_len


In [69]:
# =========================
# 3) Similarity setup (Cell 4)
#    - SBERT embeddings + cosine similarity
# =========================
import numpy as np
from tqdm.auto import tqdm

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    raise ImportError(
        "Missing dependency: sentence-transformers\n"
        "Install in your venv:\n"
        "  pip install sentence-transformers"
    )

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

def _to_str(x) -> str:
    if x is None:
        return ""
    return str(x)

def _clip_text(s: str, max_chars: int = 2000) -> str:
    # Best practice: avoid encoding massive articles (costly, noisy).
    # 1500–3000 chars is usually enough for topical similarity.
    s = s.strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars]

def encode_texts(texts, batch_size: int = 64) -> np.ndarray:
    emb = model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,  # gives unit vectors -> cosine = dot
    )
    return np.asarray(emb)

def cosine_sim(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    # since embeddings are normalized, cosine = dot product
    return (a * b).sum(axis=1)


In [70]:
# =========================
# 4) Compute similarity + split (Cell 5) — status-based thresholds ONLY
# =========================
claims = [_to_str(x) for x in full["claim_text"].tolist()]
articles_raw = [_to_str(x) for x in full["article_text"].tolist()]
articles = [_clip_text(a, max_chars=2000) for a in articles_raw]

# mark empty article_text as auto-filtered
empty_article = np.array([len(a.strip()) == 0 for a in articles], dtype=bool)

claim_emb = encode_texts(claims, batch_size=64)
article_emb = encode_texts(articles, batch_size=64)

sims = cosine_sim(claim_emb, article_emb)  # shape (N,)

full = full.copy()
full["claim_article_cosine"] = sims

# Ensure content_char_len exists (optional but useful)
if "content_char_len" not in full.columns:
    full["content_char_len"] = full["article_text"].fillna("").astype(str).str.len()


status = full["content_status"].fillna("").astype(str).str.lower()

is_full = status.isin(["full_article", "success"])
is_partial = status.eq("partial")
is_other = ~(is_full | is_partial)

# Count "unlabeled/other" statuses
unlabeled_count = int(is_other.sum())
if unlabeled_count:
    print("Unlabeled/other content_status rows:", unlabeled_count)
    print("Other statuses:", full.loc[is_other, "content_status"].value_counts().to_dict())

# Filtering logic:
# - empty -> filtered
# - partial: filter if cosine < 0.10
# - full_article: filter if cosine < 0.05
# - other: filtered (and counted above)
is_filtered = (
    empty_article
    | (is_partial.to_numpy() & (sims < THRESH_PARTIAL))
    | (is_full.to_numpy() & (sims < THRESH_FULL))
    | is_other.to_numpy()   # count + filter “other”
)

filtered_df = full[is_filtered].copy()
kept_df = full[~is_filtered].copy()

print("\nThresholds used:")
print("  partial     :", THRESH_PARTIAL)
print("  full_article:", THRESH_FULL)

print("\nCounts:")
print("  full    :", len(full))
print("  kept    :", len(kept_df))
print("  filtered:", len(filtered_df))

# quick view of worst offenders among filtered
filtered_df.sort_values("claim_article_cosine", ascending=True)[
    ["id", "content_status", "content_char_len", "claim_article_cosine", "claim_text", "article_text"]
].head(5)


Batches: 0it [00:00, ?it/s]

Batches: 0it [00:00, ?it/s]

AxisError: axis 1 is out of bounds for array of dimension 1

In [ ]:
# =========================
# 5) Save BOTH (Cell 6)
#    - kept becomes the main training file
#    - filtered kept for audit
# =========================
kept_df.to_parquet(str(KEPT_PATH), index=False)
filtered_df.to_parquet(str(FILTERED_PATH), index=False)

print("Saved kept    ->", KEPT_PATH)
print("Saved filtered->", FILTERED_PATH)


Saved kept    -> C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\fakenewsnet\fakenewsnet_false.parquet
Saved filtered-> C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\fakenewsnet\fakenewsnet_false_filtered.parquet


In [ ]:
# =========================
# 6) Sanity checks (Cell 7)
# =========================
kept_ids = set(kept_df["id"].astype(str))
filtered_ids = set(filtered_df["id"].astype(str))

overlap = kept_ids.intersection(filtered_ids)
print("kept ∩ filtered id overlap:", len(overlap))
assert len(overlap) == 0, "ERROR: overlap detected between kept and filtered!"

print("Row accounting ok:", len(kept_df) + len(filtered_df) == len(full))


kept ∩ filtered id overlap: 0
Row accounting ok: True
